In [0]:
%skip
%pip install holidays
dbutils.library.restartPython()

In [0]:
import re
import pyspark.sql.functions as F
import holidays
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from delta.tables import DeltaTable

In [0]:
df = spark.read.table("fraud_detection_project.silver_layer.transaction_events")

# Creating Temporal Features
df = df.withColumn("hour_of_day",   F.date_format(F.col("txn_timestamp"), "HH : mm : ss"))
df = df.withColumn("day_of_week",   F.dayofweek(F.col("txn_timestamp")))
df = df.withColumn("day_of_month",  F.dayofmonth(F.col("txn_timestamp")))
df = df.withColumn("month_of_year", F.month(F.col("txn_timestamp")))
df = df.withColumn("year_txn",      F.year(F.col("txn_timestamp")))
    # Features with temporal details
df = df.withColumn("is_weekend",    F.when(F.col("day_of_week") == 1, True).when(F.col("day_of_week") == 7, True).otherwise(False))
df = df.withColumn("days_since_last_txn",
                    F.datediff(F.col("txn_timestamp"),
                    F.lag(F.col("txn_timestamp")).over(Window.partitionBy("card_id").orderBy("txn_timestamp")))
                    )

df = df.withColumn(
    "hours_since_last_txn",
    F.expr("""
        CASE WHEN lag(txn_timestamp) OVER (PARTITION BY card_id ORDER BY txn_timestamp) IS NULL THEN NULL
        ELSE
            CONCAT(
                LPAD(FLOOR((unix_timestamp(txn_timestamp) - unix_timestamp(lag(txn_timestamp) OVER (PARTITION BY card_id ORDER BY txn_timestamp))) / 3600), 2, '0'), 'h ',
                LPAD(FLOOR(((unix_timestamp(txn_timestamp) - unix_timestamp(lag(txn_timestamp) OVER (PARTITION BY card_id ORDER BY txn_timestamp))) % 3600) / 60), 2, '0'), 'm ',
                LPAD(((unix_timestamp(txn_timestamp) - unix_timestamp(lag(txn_timestamp) OVER (PARTITION BY card_id ORDER BY txn_timestamp))) % 60), 2, '0'), 's'
            )
        END
    """)
)


In [0]:
# Creating  a dictionary to day_of_week and month_of_year
day_of_week_dict = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 5: "Friday", 6: "Saturday", 7: "Sunday"}
month_of_year_dict = {1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June", 7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December"}

# Mapping day_of_week and month_of_year
df = df.withColumn("day_of_week", F.create_map([F.lit(x) for x in sum(day_of_week_dict.items(), ())]).getItem(F.col("day_of_week")))
df = df.withColumn("month_of_year", F.create_map([F.lit(x) for x in sum(month_of_year_dict.items(), ())]).getItem(F.col("month_of_year")))

In [0]:
# Creating is_holiday
# Below we have the most efficient way to do this, but it is not allowed on Serverless
# txn_year = df.select("year_txn").distinct().rdd.flatMap(lambda x: x).collect()[0] 

rows_year = df.select("year_txn").distinct().collect()

txn_year = [row[0] for row in rows_year]

br_holidays = holidays.BR(years=txn_year)
list_holidays = list(br_holidays.keys())

df = df.withColumn("is_holiday",    F.when(F.col("txn_timestamp").cast("date").isin(list_holidays), True).otherwise(False))

# Creating working_days
df = df.withColumn("working_days", F.when(F.col("is_weekend") == True, False).when(F.col("is_holiday") == True, False).otherwise(True))

# Creating days_si

In [0]:
df = df.drop('txn_timestamp')
df = df.drop('txn_amount')
df = df.drop('txn_currency')
df = df.drop('txn_status')
df = df.drop('installments_count')
df = df.drop('card_id')
df = df.drop('merchant_id')
df = df.drop('is_recurring')
df = df.drop('kafka_topic')
df = df.drop('ingestion_timestamp')
df = df.drop('txn_entry_mode')
df = df.drop('card_type')

In [0]:
%skip
df.createOrReplaceTempView('df1')

In [0]:
%skip
SELECT * FROM df1 LIMIT(100);

In [0]:
#%skip
target = "fraud_detection_project.silver_layer.transaction_temporal_details"

if spark.catalog.tableExists(target):
    dt = DeltaTable.forName(spark, target)

    dt.alias("t").merge(
        df.alias("s"),
        "t.transaction_id = s.transaction_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Merge concluded.")
else:
    df.write.format("delta") \
      .option("mergeSchema", "true") \
      .saveAsTable(target)
    print("Tabel created.")

In [0]:
%sql
SELECT * FROM fraud_detection_project.silver_layer.transaction_temporal_details LIMIT(100);